
# Student Performance Intelligence System

## Complete Educational Analytics & Machine Learning Project

This notebook includes:

1. Dataset Overview
2. Data Cleaning & Preprocessing
3. Advanced Exploratory Data Analysis
4. Correlation & Heatmaps
5. Student Behavior Intelligence
6. Socioeconomic Analysis
7. Academic Risk Analysis
8. Feature Engineering
9. Predictive Machine Learning
10. Model Comparison
11. Student Segmentation
12. PCA Visualization
13. SHAP Explainability
14. Advanced Visualizations
15. Student Success Intelligence Engine

---

## Goal

Understand what factors influence student academic performance and build predictive educational intelligence systems.


In [ ]:

# Core Libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    classification_report
)

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Plotly for Kaggle
pio.renderers.default = 'iframe_connected'

# Display Settings
pd.set_option('display.max_columns', None)

print("Libraries Loaded Successfully")


In [ ]:

# Load Dataset

df = pd.read_csv('StudentPerformanceFactors (1).csv')

print("Dataset Shape:", df.shape)

df.head()


# 1. Dataset Overview

In [ ]:

df.info()

print("\nMissing Values:\n")
print(df.isnull().sum())

df.describe(include='all')


# 2. Data Cleaning & Preprocessing

In [ ]:

# Copy Dataset

data = df.copy()

# Fill missing categorical values

cat_cols = data.select_dtypes(include='object').columns

for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Fill missing numerical values

num_cols = data.select_dtypes(include=np.number).columns

for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

print("Missing Values After Cleaning:\n")
print(data.isnull().sum().sum())


# 3. Advanced Exploratory Data Analysis

In [ ]:

# Exam Score Distribution

plt.figure(figsize=(10,6))

sns.histplot(data['Exam_Score'], bins=30)

plt.title('Exam Score Distribution')
plt.xlabel('Exam Score')
plt.show()

# Hours Studied vs Exam Score

plt.figure(figsize=(10,6))

sns.scatterplot(
    x='Hours_Studied',
    y='Exam_Score',
    data=data
)

plt.title('Hours Studied vs Exam Score')
plt.show()


# 4. Correlation Heatmap

In [ ]:

numeric_data = data.select_dtypes(include=np.number)

plt.figure(figsize=(16,10))

sns.heatmap(
    numeric_data.corr(),
    cmap='coolwarm',
    annot=True
)

plt.title('Feature Correlation Heatmap')
plt.show()


# 5. Student Behavior Intelligence

In [ ]:

behavior_features = [
    'Hours_Studied',
    'Attendance',
    'Sleep_Hours',
    'Previous_Scores'
]

for feature in behavior_features:

    plt.figure(figsize=(8,5))

    sns.boxplot(
        x=data[feature]
    )

    plt.title(f'{feature} Distribution')

    plt.show()


# 6. Socioeconomic Analysis

In [ ]:

income_analysis = data.groupby('Family_Income')[
    'Exam_Score'
].mean().sort_values()

income_analysis.plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Family Income vs Average Exam Score')
plt.ylabel('Average Exam Score')
plt.show()


# 7. Academic Risk Analysis

In [ ]:

# Create Risk Label

data['at_risk'] = (
    data['Exam_Score'] < data['Exam_Score'].median()
).astype(int)

risk_counts = data['at_risk'].value_counts()

risk_counts.plot(
    kind='bar',
    figsize=(6,4)
)

plt.title('Academic Risk Distribution')
plt.show()


# 8. Feature Engineering

In [ ]:

# Study Efficiency

data['study_efficiency'] = (
    data['Exam_Score'] /
    data['Hours_Studied']
)

# Attendance Efficiency

data['attendance_efficiency'] = (
    data['Exam_Score'] /
    data['Attendance']
)

# Sleep Productivity

data['sleep_productivity'] = (
    data['Exam_Score'] /
    data['Sleep_Hours']
)

data.head()


# 9. Predictive Machine Learning

In [ ]:

# Encode Categorical Variables

ml_data = data.copy()

encoders = {}

for col in ml_data.select_dtypes(include='object').columns:

    le = LabelEncoder()

    ml_data[col] = le.fit_transform(
        ml_data[col].astype(str)
    )

    encoders[col] = le

# Features & Target

X = ml_data.drop(columns=['Exam_Score'])

y = ml_data['Exam_Score']

# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

preds = rf_model.predict(X_test_scaled)

print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("MAE:", mean_absolute_error(y_test, preds))
print("R2 Score:", r2_score(y_test, preds))


# 10. Model Comparison

In [ ]:

# Linear Regression

lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

lr_preds = lr.predict(X_test_scaled)

# Random Forest already trained

rf_preds = rf_model.predict(X_test_scaled)

comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R2 Score': [
        r2_score(y_test, lr_preds),
        r2_score(y_test, rf_preds)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, lr_preds)),
        np.sqrt(mean_squared_error(y_test, rf_preds))
    ]
})

comparison


# 11. Student Segmentation

In [ ]:

cluster_features = [
    'Hours_Studied',
    'Attendance',
    'Sleep_Hours',
    'Previous_Scores',
    'Exam_Score'
]

cluster_data = ml_data[cluster_features]

scaler = StandardScaler()

scaled_cluster = scaler.fit_transform(cluster_data)

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

ml_data['cluster'] = kmeans.fit_predict(scaled_cluster)

ml_data[['cluster']].head()


# 12. PCA Visualization

In [ ]:

pca = PCA(n_components=2)

pca_result = pca.fit_transform(scaled_cluster)

pca_df = pd.DataFrame({
    'PCA1': pca_result[:,0],
    'PCA2': pca_result[:,1],
    'cluster': ml_data['cluster']
})

fig = px.scatter(
    pca_df,
    x='PCA1',
    y='PCA2',
    color='cluster',
    title='Student Clusters using PCA'
)

fig.show()


# 13. SHAP Explainability

In [ ]:

# Optional SHAP Explainability

# Uncomment if SHAP is installed

# import shap
#
# explainer = shap.Explainer(rf_model)
# shap_values = explainer(X_test_scaled)
#
# shap.plots.beeswarm(shap_values)

print("SHAP explainability section prepared.")


# 14. Advanced Visualizations

In [ ]:

# Feature Importance

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(12,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance.head(10)
)

plt.title('Top 10 Important Features')
plt.show()


# 15. Student Success Intelligence Engine

In [ ]:

# Success Score

ml_data['success_score'] = (
    0.35 * ml_data['Exam_Score'] +
    0.25 * ml_data['Attendance'] +
    0.20 * ml_data['Hours_Studied'] +
    0.20 * ml_data['Previous_Scores']
)

top_students = ml_data[[
    'Exam_Score',
    'success_score'
]].sort_values(
    by='success_score',
    ascending=False
)

top_students.head(10)



# Final Educational Insights

## Key Learnings

- Attendance strongly impacts performance.
- Previous academic scores predict future outcomes.
- Balanced sleep improves productivity.
- Study efficiency matters more than raw study hours.
- Socioeconomic factors influence outcomes.
- Machine learning can identify at-risk students.

---

# Possible Extensions

- Streamlit Dashboard
- Power BI Dashboard
- Deep Learning Models
- XGBoost / LightGBM
- Real-time Student Monitoring
- Educational Policy Analytics

---

# End of Project
